# Part 3: Prompt Engineering Science
 
> **Central idea:** Prompting is engineering, not art. Controlled experiments reveal which components actually move the needle, and by how much.

In [146]:
# imports

import json
import time
import re
import mlflow
from chat.client import ChatClient
from utils.tracker import setup_mlflow
import pandas as pd
from IPython.display import display
import uuid
import random

In [147]:
def normalize(value):
    # Normalize the response text (e.g. remove trailing whitespace, etc)
    if value is None:
        return None
    return str(value).strip().lower()


def extract_json_object(text):
    """
    Extract the first valid JSON object from a model response.
    Prefers fenced JSON blocks when available.
    """
    if not text:
        return {}

    fenced_match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    candidate = fenced_match.group(1) if fenced_match else None

    if candidate is None:
        brace_match = re.search(r"\{.*\}", text, re.DOTALL)
        candidate = brace_match.group(0) if brace_match else None

    if candidate is None:
        return {}

    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        return {}

In [148]:
def f1_evaluator(predicted_records, ground_truth_records):
    """
    Calculates precision, recall, and F1 for exact-match field extraction.

    We treat the target as a set of key-value pairs over the schema:
    {"id" -> "sector"}.
    """
    correct = 0
    for record_id, gt_sector in ground_truth_records.items():
        pred_sector = predicted_records.get(record_id)
        if pred_sector is not None and normalize(pred_sector) == normalize(gt_sector):
            correct += 1

    total_predicted = len(predicted_records)
    total_ground_truth = len(ground_truth_records)

    precision = correct / total_predicted if total_predicted else 0.0
    recall = correct / total_ground_truth if total_ground_truth else 0.0
    f1_score = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score,
    }

In [149]:
NUM_TARGETS = 40
MIDDLE_LO = 0.40
MIDDLE_HI = 0.60

def parse_record(record_text):
    parts = record_text.strip().split(" | ")
    return {
        "id": parts[0].split(": ", 1)[1].strip(),
        "clearance": parts[1].split(": ", 1)[1].strip(),
        "sector": parts[2].split(": ", 1)[1].strip(),
    }


def get_targets_from_middle(haystack, k=NUM_TARGETS, seed=42):
    """
    Select k target records from the middle 20% of the haystack.
    Returns full records so we can derive the gold id->sector mapping.
    """
    rng = random.Random(seed)
    mid_start = int(len(haystack) * MIDDLE_LO)
    mid_end = int(len(haystack) * MIDDLE_HI)

    candidates = haystack[mid_start:mid_end]
    if k > len(candidates):
        raise ValueError("k is larger than the number of middle candidates.")

    chosen = rng.sample(candidates, k)
    return [parse_record(record) for record in chosen]

In [150]:
def build_user_input(haystack_text, target_records):
    target_ids = [record["id"] for record in target_records]
    return (
        "Read the haystack below and find the sectors for the target record ids listed below. "
        "Return as a JSON with each target Record-ID matching its corresponding Sector. "
        f"Target Record-IDs: {json.dumps(target_ids)}\n\n"
        f"Haystack:\n{haystack_text}"
    )

In [151]:
num_records = 2000
haystack_lines = []

while len(haystack_lines) < num_records:
    random_id = uuid.uuid4().hex[:8]
    record = f"Record-ID: {random_id} | Clearance: {random.randint(1,5)} | Sector: {uuid.uuid4().hex[:6]}\n"
    haystack_lines.append(record)

test_paragraph = "\n".join(haystack_lines)

In [152]:
target_records = get_targets_from_middle(haystack_lines, k=NUM_TARGETS, seed=42)
ground_truth_record = {record["id"]: record["sector"] for record in target_records}
user_input = build_user_input(test_paragraph, target_records)

## Task 3.1: PTCF Ablation Study

Instructions:
- Aim for an F1 score around 0.9

- Do NOT change the model

In [153]:
client = ChatClient(budgeting=False, model="databricks-gemma-3-12b")
registry = client.prompt_registry

In [154]:
baseline_prompt = (
    "**Input:** {{ user_input }}"
)

p_persona = (
    "**Persona:**\n"
    "You are an ultra-precise data extraction engine optimized for high-volume information retrieval "
    "and zero-hallucination structural parsing.\n\n"
)

p_task = (
    "**Task:**\n"
    "Find ALL target Record-ID values listed in the input.\n"
    "Search the entire haystack carefully.\n"
    "For every target Record-ID, extract its corresponding Sector value.\n"
    "Check every target ID before producing the final answer.\n\n"
)

p_context = (
    "**Context:**\n"
    "Use only information explicitly present in the haystack.\n"
    "Return one entry for every target Record-ID that appears.\n"
    "Do not add explanations, comments, or extra text.\n\n"
)

p_format = (
    "**Format:**\n"
    "Output ONLY a valid JSON object.\n"
    "Each key must be a Record-ID.\n"
    "Each value must be the corresponding Sector.\n"
    "No markdown.\n"
    "No code fences.\n"
    "No explanations.\n\n"
)

p_input = "**Input:** {{ user_input }}"

# Register all prompt variants
prompt_variants = {
    "baseline": baseline_prompt,
    "persona_only": p_persona + p_input,
    "task_only": p_task + p_input,
    "context_only": p_context + p_input,
    "format_only": p_format + p_input,
    "full_ptcf": p_persona + p_task + p_context + p_format + p_input,
}

# registry.create()
# registry.update()
for name, template in prompt_variants.items():
    try:
        registry.create(name, template)
    except ValueError:
        registry.update(name, template)

In [155]:
setup_mlflow("Task-3.1-PTCF-Ablation")

ablation_variants = [
    "baseline",
    "persona_only",
    "task_only",
    "context_only",
    "format_only",
    "full_ptcf"
]

ablation_results = []

MLflow experiment set: /Users/28100445@lums.edu.pk/Task-3.1-PTCF-Ablation at databricks


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


In [156]:
for variant in ablation_variants:
    print(f"Evaluating {variant}...")

    rendered_prompt = registry.render(variant, user_input=user_input)
    client.reset_conversation()
    raw_output = client.chat(rendered_prompt, stream=False)

    predicted = extract_json_object(raw_output)
    
    print("Ground Truth:", len(ground_truth_record))
    print("Predicted:", len(predicted))

    correct = sum(
        1
        for k, v in ground_truth_record.items()
        if normalize(predicted.get(k)) == normalize(v)
    )

    print("Correct:", correct)
    scores = f1_evaluator(predicted, ground_truth_record)
    token_count = client.get_token_count(raw_output)

    # start a run with MLflow
    with mlflow.start_run(run_name=f"ablation-{variant}"):
        mlflow.log_param("prompt_id", variant)
        mlflow.log_param("model", client.model)
        mlflow.log_metric("precision", scores["precision"])
        mlflow.log_metric("recall", scores["recall"])
        mlflow.log_metric("f1_score", scores["f1_score"])
        mlflow.log_metric("token_count", token_count)
        mlflow.log_text(raw_output, f"{variant}_raw_output.txt")

    ablation_results.append({
        "Variant": variant,
        "Precision": round(scores["precision"], 3),
        "Recall": round(scores["recall"], 3),
        "F1 Score": round(scores["f1_score"], 3),
        "Token Count": token_count
    })

# Display a compact summary table
ablation_df = pd.DataFrame(ablation_results)
display(ablation_df)

Evaluating baseline...
Ground Truth: 40
Predicted: 40
Correct: 29
🏃 View run ablation-baseline at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3950437384603063/runs/79d9ced34b344c3fba79c7f6b5c870a3
🧪 View experiment at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3950437384603063
Evaluating persona_only...
Ground Truth: 40
Predicted: 40
Correct: 24
🏃 View run ablation-persona_only at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3950437384603063/runs/1982299012b9490b9b8f46c00268f22e
🧪 View experiment at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3950437384603063
Evaluating task_only...
Ground Truth: 40
Predicted: 40
Correct: 31
🏃 View run ablation-task_only at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3950437384603063/runs/ec5bd53b281f48ed92522aed436b053a
🧪 View experiment at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3950437384603063
Evaluating context_only...
Ground Truth

,Variant,Precision,Recall,F1 Score,Token Count
0,baseline,0.725,0.725,0.725,546
1,persona_only,0.600,0.600,0.600,549
2,task_only,0.775,0.775,0.775,547
3,context_only,0.725,0.725,0.725,557
4,format_only,0.650,0.650,0.650,554
5,full_ptcf,0.825,0.825,0.825,550


#### Analysis questions

1. Which single PTCF component had the largest marginal F1 gain when added to the simplest baseline? Report the exact scores before and after the change.
2. Did the full PTCF prompt always beat two-component prompts? Identify any cases where adding a component hurt performance and give a concrete explanation.
3. Did the model ever produce entity types outside the schema, such as person names or locations? Which prompt component reduced that behavior the most?
4. Your evaluator uses exact string matching. How would the scores change under fuzzy matching, and is exact-match a fair metric for this task? State the tradeoff clearly.

## Task 3.2: Prompting Strategies Comparison

In [157]:
# Build the three strategy prompts from the best-performing PTCF variant found in Task 3.1
# (This cell assumes ablation_results has already been populated.)

ptcf_candidates = [row for row in ablation_results]
best_ptcf_variant = max(ptcf_candidates, key=lambda row: row["F1 Score"])["Variant"]
best_ptcf_template = prompt_variants[best_ptcf_variant]


In [158]:
few_shot_examples = """
**Example 1:**
Input: Find the Sector for Record-ID: a1b2c3d4.
Output: {"a1b2c3d4": "9f8e7d"}

**Example 2:**
Input: Find the Sector for Record-ID: 55667788.
Output: {"55667788": "112233"}

"""

chain_of_thought = """
Before answering, think step by step: scan the haystack line by line, locate each target
Record-ID, and note its Sector value. Show this reasoning under an "Explanation:" heading,
then give the final answer as a fenced JSON block under a "Final Answer:" heading.
"""

In [159]:
strategy_zero_shot = best_ptcf_template
strategy_few_shot = best_ptcf_template.replace("**Input:**", few_shot_examples + "\n**Input:**")
strategy_cot = best_ptcf_template.replace("**Input:**", chain_of_thought + "\n**Input:**")

try:
    registry.create("strategy_zero_shot", strategy_zero_shot)
    registry.create("strategy_few_shot", strategy_few_shot)
    registry.create("strategy_cot", strategy_cot)
except ValueError:
    registry.update("strategy_zero_shot", strategy_zero_shot)
    registry.update("strategy_few_shot", strategy_few_shot)
    registry.update("strategy_cot", strategy_cot)

In [160]:
setup_mlflow("Task-3.2-Prompt-Strategies")

strategies = [
    "strategy_zero_shot",
    "strategy_few_shot",
    "strategy_cot"
]

strategy_results = []

MLflow experiment set: /Users/28100445@lums.edu.pk/Task-3.2-Prompt-Strategies at databricks


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


In [161]:
for strategy in strategies:
    print(f"Evaluating {strategy}...")

    rendered_prompt = registry.render(strategy, user_input=user_input)

    client.reset_conversation()
    start_time = time.time()
    raw_output = client.chat(rendered_prompt, stream=False)
    latency = time.time() - start_time

    predicted = extract_json_object(raw_output)
    scores = f1_evaluator(predicted, ground_truth_record)
    token_count = client.get_token_count(raw_output)

    with mlflow.start_run(run_name=f"strategy-{strategy}"):
        mlflow.log_param("strategy", strategy)
        mlflow.log_metric("f1_score", scores["f1_score"])
        mlflow.log_metric("token_count", token_count)
        mlflow.log_metric("latency_sec", latency)

    strategy_results.append({
        "Strategy": strategy,
        "F1 Score": round(scores["f1_score"], 3),
        "Token Count": token_count,
        "Latency (s)": round(latency, 3)
    })

# Display a compact summary table
strategy_df = pd.DataFrame(strategy_results)
display(strategy_df)

Evaluating strategy_zero_shot...
🏃 View run strategy-strategy_zero_shot at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3950437384603064/runs/086ed0dc5af54301938c1d2aa75e5bc3
🧪 View experiment at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3950437384603064
Evaluating strategy_few_shot...
🏃 View run strategy-strategy_few_shot at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3950437384603064/runs/6cf569885cc341098c04192faa579fc6
🧪 View experiment at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3950437384603064
Evaluating strategy_cot...
🏃 View run strategy-strategy_cot at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3950437384603064/runs/595cb097fed4463f8b28ab6a4fde6028
🧪 View experiment at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3950437384603064


,Strategy,F1 Score,Token Count,Latency (s)
0,strategy_zero_shot,0.725,549,11.341
1,strategy_few_shot,0.750,549,11.750
2,strategy_cot,0.775,552,12.005


#### Analysis questions

1. Rank the three strategies by F1 score. Does the ranking match your expectation? State any surprises.
2. Did CoT help, hurt, or make no measurable difference on this extraction task? Explain what that implies about when CoT is useful.
3. Few-shot examples consume context. At what point would adding more examples start to reduce performance because the document itself no longer fits comfortably? Explain how you would test that threshold.
4. You used 2-shot. Would 5-shot likely improve, stay flat, or degrade? Give a reasoned prediction and describe the experiment you would run to verify it.